# Data Preparation: FIDS30 + External Test Sets

Three datasets used in Unit 3:
- **FIDS30** (fruits) — \"normal\" data for training the autoencoder. Downloaded from vicos.si.
- **Near-OOD (nuts)** — 33 images each of almonds, cashews, pistachios. Pre-sampled in `external_data/NearOOD/`.
- **Far-OOD (UC Merced satellite)** — 5 images per class × 21 classes. Pre-sampled in `external_data/UC_Merced/`.

The two OOD sets are pre-committed to the repo so no downloads are needed for them.

## 1. Download and Prepare FIDS30

In [ ]:
import urllib.request
import zipfile
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_URL = "https://data.vicos.si/datasets/FIDS30/FIDS30.zip"
ZIP_PATH = Path("FIDS30.zip")
EXTRACT_DIR = Path("data")

if not (EXTRACT_DIR / "FIDS30").exists():
    if not ZIP_PATH.exists():
        print("Downloading FIDS30 dataset...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
        print(f"Downloaded: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)")
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"Extracted to {EXTRACT_DIR}/")
else:
    print("FIDS30 already present.")

# Split into train/val/test
SOURCE = Path("data/FIDS30")
TARGET = Path("PrepData/FIDS30")

all_images, all_labels = [], []
classes = sorted([d.name for d in SOURCE.iterdir() if d.is_dir()])
for cls in classes:
    imgs = [p for p in (SOURCE / cls).glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]]
    all_images.extend(imgs)
    all_labels.extend([cls] * len(imgs))

train_imgs, temp_imgs, train_labels, temp_labels = train_test_split(
    all_images, all_labels, test_size=0.4, stratify=all_labels, random_state=42)
val_imgs, test_imgs, val_labels, test_labels = train_test_split(
    temp_imgs, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42)

if TARGET.exists():
    shutil.rmtree(TARGET)

for split_name, (imgs, labels) in [("Training", (train_imgs, train_labels)),
                                    ("Validation", (val_imgs, val_labels)),
                                    ("Test", (test_imgs, test_labels))]:
    for img_path, cls in zip(imgs, labels):
        dest = TARGET / split_name / cls
        dest.mkdir(parents=True, exist_ok=True)
        os.symlink(img_path.absolute(), dest / img_path.name)
    print(f"  FIDS30 {split_name}: {len(imgs)} images")

print(f"FIDS30 classes: {len(classes)}")

## 2. Verify External Test Sets

The Near-OOD (nuts) and Far-OOD (UC Merced) sets live under `external_data/` and are already committed to the repo.

In [ ]:
EXTERNAL = Path("external_data")
NEAR_OOD_DIR = EXTERNAL / "NearOOD"
UC_DIR = EXTERNAL / "UC_Merced"

for name, d in [("Near-OOD (nuts)", NEAR_OOD_DIR), ("Far-OOD (UC Merced)", UC_DIR)]:
    if not d.exists():
        print(f"  {name}: MISSING at {d}")
        continue
    classes = sorted([c.name for c in d.iterdir() if c.is_dir()])
    total = sum(1 for _ in d.rglob("*.jpg"))
    print(f"  {name}: {total} images across {len(classes)} classes")
    for cls in classes:
        n = sum(1 for _ in (d / cls).glob("*"))
        print(f"    {cls}: {n}")

In [ ]:
import matplotlib.pyplot as plt
from torchvision import datasets

fig, axes = plt.subplots(3, 5, figsize=(15, 9))

# FIDS30 samples
fids_ds = datasets.ImageFolder(str(TARGET / "Training"))
for j in range(5):
    idx = fids_ds.targets.index(j)
    axes[0, j].imshow(fids_ds[idx][0])
    axes[0, j].set_title(fids_ds.classes[j], fontsize=9)
    axes[0, j].axis('off')
axes[0, 0].set_ylabel("FIDS30\n(normal)", fontsize=11)

# Near-OOD (nuts) samples
near_ds = datasets.ImageFolder(str(NEAR_OOD_DIR))
for j in range(min(5, len(near_ds.classes))):
    idx = near_ds.targets.index(j)
    axes[1, j].imshow(near_ds[idx][0])
    axes[1, j].set_title(near_ds.classes[j], fontsize=9)
    axes[1, j].axis('off')
for j in range(len(near_ds.classes), 5):
    axes[1, j].axis('off')
axes[1, 0].set_ylabel("Near-OOD\n(nuts)", fontsize=11)

# UC Merced samples
uc_ds = datasets.ImageFolder(str(UC_DIR))
for j in range(5):
    idx = uc_ds.targets.index(j)
    axes[2, j].imshow(uc_ds[idx][0])
    axes[2, j].set_title(uc_ds.classes[j], fontsize=9)
    axes[2, j].axis('off')
axes[2, 0].set_ylabel("UC Merced\n(far-OOD)", fontsize=11)

plt.suptitle("Normal (fruits) vs Near-OOD (nuts) vs Far-OOD (satellite)", fontsize=13)
plt.tight_layout()
plt.show()